# 📊 Return Probability Prediction + Rule-Based Action Engine

This notebook builds a predictive model designed to score e-commerce orders on their **probability of return**. 

Instead of making binary Yes/No classifications, the ML model produces a **Return Risk Score (0 to 1)**. This prediction is fed into a **Rule-Based Decision Layer** that dictates the ultimate business interaction (e.g., block COD, restrict shipment speed, alert the customer) based not only on probability but also on customer behavior traits.

---
## 1. 🧾 Problem Statement

* **Objective:** Predict the **continuous probability of a product being returned**, rather than a strict binary classification.
* **Business Impact:** 
  * Reduce logistics and reverse-shipping losses.
  * Optimize inventory visibility.
  * Improve customer satisfaction via targeted friction limits.
* **Architecture:** 
  > **Note:** The model does NOT take autonomous decisions — it only evaluates risk. The **Decision Engine (Rule-Based System)** dictates the final downstream actions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style="whitegrid")

## 2. 📂 Dataset Selection

We are using our synthetic multi-domain e-commerce dataset generated to accurately simulate supply chain logistics, customer profiles, and product characteristics.

In [ ]:
# Load the dataset
# Adjusting path to point to the 'data/' directory generated earlier
data_path = '../data/final_combined_data.csv'
df = pd.read_csv(data_path)

print(f"Dataset successfully loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

## 3. 📊 Dataset Description & 4. 📘 Data Documentation

The dataset merges `products`, `customers`, `orders`, `logistics`, and `returns`.
* **Input Features:** Include multi-domain data covering delivery timings, pricing, category rates, and historical return rates.
* **Target Feature:** We will use `is_returned` as the ground truth. Since we are using a **Regression** model on a binary target, the output naturally models the **`return_probability`** in the range [0.0, 1.0].

In [ ]:
# Display primary column info
display(df.info())

# Preview top features
display(df[['is_returned', 'delivery_delay', 'discount_percentage', 'avg_rating', 'overall_return_rate', 'frequent_return_flag']].head())

## 5. 📥 Raw Data Exploration

Let's look at the basic statistical distribution of our underlying numeric features and the baseline layout of returns.

In [ ]:
# Descriptive statistics
display(df.describe())

# Visualizing baseline return classes to understand the base probability
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='is_returned', palette='viridis')
plt.title('Actual Returns Distribution (Target)')
plt.show()

## 6. 🧹 Data Cleaning & 7. 📉 Outlier Analysis

Because we utilize an actively maintained synthetic generator, there are no missing values. However, we'll strip out non-predictive baseline IDs to avoid overfitting or dimensionality problems. Furthermore, we cap extreme delays effectively.

In [ ]:
# Drop purely categorical IDs and metadata that aren't useful for regression
drop_cols = [
    'order_id', 'customer_id', 'product_id', 'order_date', 
    'product_name', 'category_return_rate', 'product_return_rate'
]

# Ensure we only drop if they exist
cols_to_drop = [c for c in drop_cols if c in df.columns]
df_clean = df.drop(columns=cols_to_drop)

# Outlier Handling: Cap severe delivery delays to 15 days to stabilize prediction space
df_clean['delivery_delay'] = np.clip(df_clean['delivery_delay'], a_min=-5, a_max=15)

print(f"Data cleaned. Columns remaining: {df_clean.shape[1]}")

## 8. 🔄 Data Transformation & 9. 🛠️ Feature Engineering

We will encode the categorical fields and create targeted interaction features (e.g., `Delay × Category Risk`) to capture deeper non-linear behaviors, pushing the ML's ability to learn real-world signals.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Interaction Feature: High value delayed shipments
df_clean['high_value_delayed'] = df_clean['final_price'] * df_clean['delivery_delay']

# Encode categorical variables using dummy variables
cat_columns = df_clean.select_dtypes(include=['object', 'bool']).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=cat_columns, drop_first=True)

# Separate inputs (X) and target (y)
# Predict probability directly using the 0/1 target mapped via Regressor
X = df_encoded.drop(columns=['is_returned', 'return_days_after_delivery', 'return_reason_Wrong Item', 'return_reason_Damaged', 'return_reason_Changed Mind', 'return_reason_Late Delivery'], errors='ignore')
y = df_encoded['is_returned']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 10. 🤖 Model Development (Regression Only)

We employ a **RandomForestRegressor** directly on our target variable. Because the ground truth is `[0, 1]`, the continuous predictions emitted by the ensemble's decision trees will output a distinct decimal representing our target **`return_probability`**.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Train Regressor
model = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_train_scaled, y_train)

# Predict probabilities
predictions = model.predict(X_test_scaled)
predictions = np.clip(predictions, 0, 1) # Ensure bounded explicitly between 0% and 100%

# Evaluate metrics
print("--- 📉 ML Regression Metrics ---")
print(f"R-Squared (R²): {r2_score(y_test, predictions):.4f}")
print(f"RMSE          : {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"MAE           : {mean_absolute_error(y_test, predictions):.4f}")

---
## ⚙️ Rule-Based Decision Layer (Business Logic)

This is the critical intersection between Data Science and Business Operations. 
We take the predictions from our ML Regressor and inject **User Behavior Context** (like `overall_return_rate` and `frequent_return_flag`) to formulate direct supply-chain interventions.

### Hard Coded Actions:
* **`probability < 0.20`** → ✅ No action
* **`0.20 ≤ probability < 0.50`** → ⚠️ Show warning / restrict discounts
* **`0.50 ≤ probability < 0.75`** → 🚚 Avoid fast shipping / verify order
* **`probability ≥ 0.75`** → ❌ Block COD / Require prepayment

*(Personalized override: Loyal customers with moderate risk are shifted down a penalty tier!)*

In [ ]:
# Re-attach useful behavioral indices to our predicted output
results_df = X_test.copy()
results_df['target_actual'] = y_test
results_df['return_probability'] = predictions

# Map out the Decision Function
def decision_engine(row):
    prob = row['return_probability']
    
    # Behavioral flags (if exact boolean is converted to float by OHE, we treat > 0.5 as True)
    historical_return_rate = row.get('overall_return_rate', 0)
    is_frequent = row.get('frequent_return_flag_True', 0) > 0.5
    
    # Tier 1: Low Risk
    if prob < 0.20:
        return "✅ Allow Order - No Action Needed"
        
    # Tier 2: Medium Risk
    elif 0.20 <= prob < 0.50:
        # Override for loyal/low-historical-return customers
        if historical_return_rate <= 0.15 and not is_frequent:
             return "✅ Allow Order - Loyal Override"
        return "⚠️ Restrict Discounts / Show Return Warning"
        
    # Tier 3: High Risk
    elif 0.50 <= prob < 0.75:
        # Override for known high-return abusers
        if is_frequent:
             return "❌ Block COD / Require Prepayment (Frequent Returner Escalation)"
        return "🚚 Avoid Expedited Shipping (Standard Only) / Order Verification"
        
    # Tier 4: Critical Risk
    else:
        return "❌ Block COD / Require Full Prepayment"

# Apply Business Rules
results_df['business_action'] = results_df.apply(decision_engine, axis=1)

# Display Sample Operations
display_cols = ['return_probability', 'target_actual', 'overall_return_rate', 'business_action']
cols_to_print = [c for c in display_cols if c in results_df.columns]

# Random sample showing the diversity of engine decisions
print("\n--- 💼 Sample Business Decisions Generated ---")
pd.set_option('display.max_colwidth', None)
display(results_df[cols_to_print].sample(15, random_state=42))